# HW01-B — SQL, Latency, and Metabase

The business team does not care that your notebook works. They want a dashboard that opens fast.

Here you connect to shared Postgres, write SQL, measure latency, create a materialized view in your own schema, and build a Metabase dashboard.

## Submission discipline

This is individual work.

Work locally. Push to GitHub. Use the shared server services through URLs and credentials. Do not SSH into the server.

Do not commit `.env`, `.venv/`, passwords.

## Credentials and shared services

Credentials, service URLs, and connection details are provided on the HW page.

Use those exact values. Everyone must work against the same QBC12 database snapshot and the same shared Metabase/Airflow services.

Do not paste credentials into notebook markdown. Do not commit `.env` files. Do not screenshot passwords.


## Useful references

- PostgreSQL `EXPLAIN`: https://www.postgresql.org/docs/current/sql-explain.html
- PostgreSQL using `EXPLAIN`: https://www.postgresql.org/docs/current/using-explain.html
- Metabase questions: https://www.metabase.com/docs/latest/questions/introduction
- Metabase dashboards: https://www.metabase.com/docs/latest/dashboards/introduction

if you cannot open any one of these contact me : Bale (arianaghamohseni, image of a scared chicken), or Telegram (@arianaghamohseni)

## What to avoid

- `select *` in dashboard queries.
- Creating objects in `core`. You do not own `core`.
- Optimizing without runtime measurements.
- Making Metabase run a massive join every time someone opens the dashboard.

In [25]:
import os, re, time
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text

for path in ['sql', 'reports', 'screenshots']:
    Path(path).mkdir(exist_ok=True)

DB_HOST = os.getenv('QBC12_DB_HOST', '185.50.38.163')    # this is in the excel file give in Quera
DB_PORT = os.getenv('QBC12_DB_PORT', '32112')
DB_NAME = os.getenv('QBC12_DB_NAME', 'qbc12_airbnb')
DB_USER = os.getenv('QBC12_DB_USER', '') or input('DB user: ').strip()
DB_PASSWORD = os.getenv('QBC12_DB_PASSWORD', '') or input('DB password: ').strip()
STUDENT_ID = os.getenv('QBC12_STUDENT_ID', '') or DB_USER.replace('student_', '')

safe_student = re.sub(r'[^a-zA-Z0-9_]', '_', STUDENT_ID.lower())
STUDENT_SCHEMA = f'student_{safe_student}'
engine = create_engine(f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}', pool_pre_ping=True)
with engine.begin() as conn:
    conn.execute(text("SET statement_timeout = '30s'"))
    version = conn.execute(text('select version()')).scalar()
STUDENT_SCHEMA, version[:80]

OperationalError: (psycopg2.OperationalError) connection to server at "185.50.38.163", port 32112 failed: Connection timed out (0x0000274C/10060)
	Is the server running on that host and accepting TCP/IP connections?

(Background on this error at: https://sqlalche.me/e/20/e3q8)

## 1. Inspect before querying

You are not allowed to write the final query blind. Check columns and row counts first.

In [ ]:
columns_sql = '''
select table_schema, table_name, column_name, data_type
from information_schema.columns
where table_schema = 'core'
  and table_name in ('listing', 'calendar_day', 'review')
order by table_name, ordinal_position;
'''
pd.read_sql(columns_sql, engine)

,table_schema,table_name,column_name,data_type
0,core,calendar_day,listing_id,bigint
1,core,calendar_day,date,date
2,core,calendar_day,available,boolean
3,core,calendar_day,price,numeric
4,core,calendar_day,adjusted_price,numeric
5,core,calendar_day,minimum_nights,integer
6,core,calendar_day,maximum_nights,integer
7,core,listing,listing_id,bigint
8,core,listing,host_id,bigint
9,core,listing,neighbourhood_id,integer


In [ ]:
row_count_sql = '''
select 'core.listing' as table_name, count(*) as rows from core.listing
union all select 'core.calendar_day', count(*) from core.calendar_day
union all select 'core.review', count(*) from core.review;
'''
pd.read_sql(row_count_sql, engine)

,table_name,rows
0,core.listing,10480
1,core.calendar_day,3825200
2,core.review,501084


## 2. Create your sandbox schema

This is the only place you write database objects.

In [ ]:
# TODO 2.1
# Create your schema if it does not exist.
# Schema name is STUDENT_SCHEMA.

check_schema_sql = f"SELECT schema_name FROM information_schema.schemata WHERE schema_name = '{STUDENT_SCHEMA}'"
result = pd.read_sql(check_schema_sql, engine)
if not result.empty:
    print(f"Schema {STUDENT_SCHEMA} already exists!")
else:
    with engine.begin() as conn:
        conn.execute(text(f"CREATE SCHEMA {STUDENT_SCHEMA}"))
    print(f"✅ Schema {STUDENT_SCHEMA} created!")

Schema student_arshia_ataei already exists!


## 3. Build baseline SQL in pieces

Do not write one giant query first. Build the CTEs, test them, then combine.

In [ ]:
# TODO 3.1
# Write calendar_30_sql.
# Required output: listing_id, avg_calendar_price_30, availability_30_rate.

# CTE to extract only the last 30 days from the calendar table
calendar_30_sql = '''
-- CTE to filter last 30 days from 3.8M calendar_day rows
WITH last_30_days AS (
    SELECT listing_id, available, price
    FROM core.calendar_day
    WHERE 30 >= CURRENT_DATE - date
)
-- Calculate average price and availability rate per listing
SELECT 
    listing_id,
    AVG(price) AS avg_calendar_price_30,
    AVG(CASE WHEN available = 't' THEN 1.0 ELSE 0.0 END) AS availability_30_rate
FROM last_30_days
GROUP BY listing_id
'''

In [ ]:
# TODO 3.2
# Write review_counts_sql.
# Required output: listing_id, total_reviews.

# CTE to count total reviews for each listing
review_counts_sql = '''
-- CTE to count total reviews for each listing
SELECT listing_id, COUNT(*) AS total_reviews
FROM core.review
GROUP BY listing_id
'''

In [ ]:
# TODO 3.3
# Combine the CTEs with core.listing into baseline_sql.
# Required output:
# neighbourhood, num_listings, avg_price, median_price,
# avg_minimum_nights, total_reviews, reviews_per_listing, availability_30_rate.

# Combine listing data with calendar and review CTEs using LEFT JOINs, then aggregate by neighbourhood
# سر این دهنم سرویس شد!
import time
import pandas as pd
from pathlib import Path 
baseline_sql = '''
-- Step 1: Filter last 30 days of calendar
WITH calendar_30 AS (
    SELECT listing_id, available, price
    FROM core.calendar_day
    WHERE 30 >= CURRENT_DATE - date
),
-- Step 2: Count reviews per listing
review_counts AS (
    SELECT listing_id, COUNT(*) AS total_reviews
    FROM core.review
    GROUP BY listing_id
)
-- Step 3: Join listing with calendar and reviews, then aggregate by neighbourhood
SELECT 
    core.listing.neighbourhood_id,
    COUNT(core.listing.listing_id) AS num_listings,
    AVG(core.listing.listing_price) AS avg_price,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY core.listing.listing_price) AS median_price,
    AVG(core.listing.minimum_nights) AS avg_minimum_nights,
    COALESCE(SUM(review_counts.total_reviews), 0) AS total_reviews,
    CASE 
        WHEN COUNT(core.listing.listing_id) > 0 
        THEN COALESCE(SUM(review_counts.total_reviews), 0)::float / COUNT(core.listing.listing_id)
        ELSE 0 
    END AS reviews_per_listing,
    COALESCE(AVG(CASE WHEN calendar_30.available = 't' THEN 1.0 ELSE 0.0 END), 0) AS availability_30_rate
FROM core.listing
LEFT JOIN calendar_30 ON core.listing.listing_id = calendar_30.listing_id
LEFT JOIN review_counts ON core.listing.listing_id = review_counts.listing_id
GROUP BY core.listing.neighbourhood_id
ORDER BY num_listings DESC
'''
Path('sql/01_baseline_neighbourhood_summary.sql').write_text(baseline_sql)

1158

In [ ]:
def timed_read_sql(sql: str, repeats: int = 3):
    times = []
    last_df = None
    for _ in range(repeats):
        start = time.perf_counter()
        last_df = pd.read_sql(sql, engine)
        times.append(time.perf_counter() - start)
    return last_df, times

baseline_df, baseline_times = timed_read_sql(baseline_sql, repeats=3)
baseline_df.head(), baseline_times

(   neighbourhood_id  num_listings   avg_price  median_price  \
 0                22        242272  271.282258         240.0   
 1                 2        161738  315.880519         245.5   
 2                20        160666  280.465331         250.0   
 3                 1        123682  307.724138         240.0   
 4                 4         98624  255.040816         214.5   
 
    avg_minimum_nights  total_reviews  reviews_per_listing  \
 0            3.949668      8408902.0            34.708518   
 1            4.009114     14270464.0            88.231980   
 2            5.464554      6154754.0            38.307756   
 3            4.888407     10304466.0            83.314193   
 4            4.240489      3573512.0            36.233696   
 
    availability_30_rate  
 0              0.206173  
 1              0.288429  
 2              0.245758  
 3              0.331067  
 4              0.213173  ,
 [2.8934281000110786, 2.7660000000032596, 2.659351300011622])

## 4. Read the query plan

`EXPLAIN ANALYZE` actually runs the query. Look for big scans, expensive joins, and repeated work.

In [ ]:
# TODO 4.1
# Run EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT) on baseline_sql.
# Save the plan to reports/baseline_explain_analyze.txt.

# Run EXPLAIN ANALYZE to create the query execution plan and identify possible weaknesses...
explain_sql = f"EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT) {baseline_sql}"

with engine.connect() as conn:
    result = conn.execute(text(explain_sql))
    plan = "\n".join([row[0] for row in result]) # to have a more clear view

Path('reports/baseline_explain_analyze.txt').write_text(plan)
print("Plan saved!")

Plan saved!


In [ ]:
# TODO 4.2
# Write reports/explain_notes.md with 3 specific observations from the plan.
# Do not write vague nonsense like 'the query is slow'.

notes = """# EXPLAIN ANALYZE Observations

1. Sequential scan on calendar_day: 3.8M rows scanned, 2.4M removed by filter.
2. External merge sort for GROUP BY: 64MB disk used, 2.1s for sorting.(Theres 1.4 MILION rows to sort!!)
3. No indexes on calendar_day.date or listing.neighbourhood_id.

"""
Path('reports/explain_notes.md').write_text(notes)
print("Notes saved!")

Notes saved!


## 5. Create a materialized view

Metabase should read from a prepared object, not a fresh monster join.

In [ ]:
# TODO 5.1
# Create optimized_sql.
# It should create student_<you>.mv_airbnb_neighbourhood_summary and at least two indexes.

optimized_sql = f'''
-- Drop existing materialized view if theres one
DROP MATERIALIZED VIEW IF EXISTS "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary;

-- Create materialized view with pre-aggregated neighbourhood data
CREATE MATERIALIZED VIEW "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary AS
-- Step 1: Last 30 days calendar
WITH calendar_30 AS (
    SELECT listing_id, available, price
    FROM core.calendar_day
    WHERE 30 >= CURRENT_DATE - date
),
-- Step 2: Review counts per listing
review_counts AS (
    SELECT listing_id, COUNT(*) AS total_reviews
    FROM core.review
    GROUP BY listing_id
)
-- Step 3: Join and aggregate
SELECT 
    core.listing.neighbourhood_id,
    COUNT(core.listing.listing_id) AS num_listings,
    AVG(core.listing.listing_price) AS avg_price,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY core.listing.listing_price) AS median_price,
    AVG(core.listing.minimum_nights) AS avg_minimum_nights,
    COALESCE(SUM(review_counts.total_reviews), 0) AS total_reviews,
    CASE 
        WHEN COUNT(core.listing.listing_id) > 0 
        THEN COALESCE(SUM(review_counts.total_reviews), 0)::float / COUNT(core.listing.listing_id)
        ELSE 0 
    END AS reviews_per_listing,
    COALESCE(AVG(CASE WHEN calendar_30.available = 't' THEN 1.0 ELSE 0.0 END), 0) AS availability_30_rate
FROM core.listing
LEFT JOIN calendar_30 ON core.listing.listing_id = calendar_30.listing_id
LEFT JOIN review_counts ON core.listing.listing_id = review_counts.listing_id
GROUP BY core.listing.neighbourhood_id
ORDER BY num_listings DESC;

-- Index for fast lookup by neighbourhood name
CREATE INDEX idx_mv_neighbourhood ON "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary (neighbourhood);

-- Index for sorting by popularity
CREATE INDEX idx_mv_num_listings ON "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary (num_listings DESC);
'''

Path('sql/02_create_materialized_view.sql').write_text(optimized_sql)
print("SQL saved!")

SQL saved!


In [ ]:
# TODO 5.2
# Execute optimized_sql statement by statement.

with engine.begin() as conn:
    conn.execute(text(f'DROP MATERIALIZED VIEW IF EXISTS "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary'))
    conn.execute(text(f'''
        CREATE MATERIALIZED VIEW "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary AS
        WITH calendar_30 AS (
            SELECT listing_id, available, price
            FROM core.calendar_day
            WHERE date >= CURRENT_DATE - INTERVAL '30 days'
        ),
        review_counts AS (
            SELECT listing_id, COUNT(*) AS total_reviews
            FROM core.review
            GROUP BY listing_id
        )
        SELECT 
            core.listing.neighbourhood_id AS neighbourhood,
            COUNT(core.listing.listing_id) AS num_listings,
            AVG(core.listing.listing_price) AS avg_price,
            PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY core.listing.listing_price) AS median_price,
            AVG(core.listing.minimum_nights) AS avg_minimum_nights,
            COALESCE(SUM(review_counts.total_reviews), 0) AS total_reviews,
            CASE 
                WHEN COUNT(core.listing.listing_id) > 0 
                THEN COALESCE(SUM(review_counts.total_reviews), 0)::float / COUNT(core.listing.listing_id)
                ELSE 0 
            END AS reviews_per_listing,
            COALESCE(AVG(CASE WHEN calendar_30.available = 't' THEN 1.0 ELSE 0.0 END), 0) AS availability_30_rate,
            COALESCE(AVG(CASE WHEN calendar_30.available = 't' THEN 1.0 ELSE 0.0 END), 0) * 365 AS availability_365_rate
        FROM core.listing
        LEFT JOIN calendar_30 ON core.listing.listing_id = calendar_30.listing_id
        LEFT JOIN review_counts ON core.listing.listing_id = review_counts.listing_id
        GROUP BY core.listing.neighbourhood_id
        ORDER BY num_listings DESC
    '''))
    conn.execute(text(f'CREATE INDEX IF NOT EXISTS idx_mv_neighbourhood ON "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary (neighbourhood)'))
    conn.execute(text(f'CREATE INDEX IF NOT EXISTS idx_mv_num_listings ON "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary (num_listings DESC)'))

In [ ]:
check_sql = f'''select * from "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary order by num_listings desc limit 10;'''
pd.read_sql(check_sql, engine)

,neighbourhood,num_listings,avg_price,median_price,avg_minimum_nights,total_reviews,reviews_per_listing,availability_30_rate,availability_365_rate
0,22,242272,271.282258,240.0,3.949668,8408902.0,34.708518,0.206173,75.253228
1,2,161738,315.880519,245.5,4.009114,14270464.0,88.231980,0.288429,105.276744
2,20,160666,280.465331,250.0,5.464554,6154754.0,38.307756,0.245758,89.701773
3,1,123682,307.724138,240.0,4.888407,10304466.0,83.314193,0.331067,120.839370
4,4,98624,255.040816,214.5,4.240489,3573512.0,36.233696,0.213173,77.808241
5,21,98490,309.583908,238.0,3.993197,3945496.0,40.059864,0.248025,90.529191
6,13,87636,251.659509,222.0,4.061162,2740032.0,31.266055,0.203569,74.302798
7,14,73298,204.674330,184.0,6.076782,2205774.0,30.093236,0.168627,61.548746
8,18,64990,370.496032,196.5,3.336082,3031482.0,46.645361,0.194753,71.084859
9,3,58424,216.597403,195.0,4.399083,1874392.0,32.082569,0.200534,73.194920


## 6. Compare latency

Numbers or it did not happen.

In [ ]:
dashboard_sql = f'''
select neighbourhood, num_listings, avg_price, median_price,
       total_reviews, reviews_per_listing, availability_30_rate, availability_365_rate
from "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary
order by num_listings desc;
'''
dashboard_df, dashboard_times = timed_read_sql(dashboard_sql, repeats=5)
perf = pd.DataFrame([
    {'query': 'baseline_direct_query', 'best_seconds': min(baseline_times), 'avg_seconds': sum(baseline_times)/len(baseline_times)},
    {'query': 'materialized_view_read', 'best_seconds': min(dashboard_times), 'avg_seconds': sum(dashboard_times)/len(dashboard_times)},
])
perf['speedup_vs_baseline_best'] = perf.loc[0, 'best_seconds'] / perf['best_seconds']
perf

,query,best_seconds,avg_seconds,speedup_vs_baseline_best
0,baseline_direct_query,2.659351,2.772926,1.000000
1,materialized_view_read,0.247491,0.345453,10.745227


## 7. Metabase dashboard

Open the shared Metabase URL and create:

```text
QBC12 HW01 - <your-github-username> - Airbnb Ops
```

Required cards:

1. listings by neighbourhood
2. average price by neighbourhood
3. review activity by neighbourhood
4. availability rate by neighbourhood
5. top neighbourhoods table

Screenshot path:

```text
screenshots/metabase_dashboard.png
```

In [ ]:
# TODO 7.1
# Write reports/hw01_b_sql_performance.md.
# Include schema, runtimes, speedup, what changed, and Metabase screenshot/link.

report = f"""# HW01-B SQL Performance Report

## Schema
- Student schema: `{STUDENT_SCHEMA}`
- Materialized view: `mv_airbnb_neighbourhood_summary`

## Runtimes
- Baseline: best {min(baseline_times):.2f}s
- Materialized view: best {min(dashboard_times):.3f}s

## Speedup
- {min(baseline_times)/min(dashboard_times):.1f}x faster

## Dashboard
- Name: QBC12 HW01 - arshiaata-blue - Airbnb Ops
- Screenshot: screenshots/metabase_dashboard.png
"""
Path('reports/hw01_b_sql_performance.md').write_text(report)
print("Report saved!")

Report saved!


In [ ]:
for file in ['sql/01_baseline_neighbourhood_summary.sql','sql/02_create_materialized_view.sql','reports/baseline_explain_analyze.txt','reports/explain_notes.md','reports/hw01_b_sql_performance.md']:
    assert Path(file).exists(), f'Missing {file}'
assert len(dashboard_df) > 0
perf

,query,best_seconds,avg_seconds,speedup_vs_baseline_best
0,baseline_direct_query,2.659351,2.772926,1.000000
1,materialized_view_read,0.260381,0.272002,10.213304


## Commit

```bash
git add sql reports screenshots notebooks
git commit -m "HW01-B SQL performance and Metabase dashboard"
```